In [5]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)

In [6]:
sys.path.append(os.path.abspath("../preprocessing"))

from preprocess_churn import load_and_prepare_data

data = load_and_prepare_data()

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
preprocessor = data["preprocessor"]

X_train.shape, X_test.shape

((4504, 27), (1126, 27))

In [8]:
baseline_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

baseline_rf.fit(X_train, y_train)

baseline_pred = baseline_rf.predict(X_test)
baseline_proba = baseline_rf.predict_proba(X_test)[:, 1]

baseline_results = {
    "model": "Baseline Random Forest",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred),
    "recall": recall_score(y_test, baseline_pred),
    "f1_score": f1_score(y_test, baseline_pred),
    "roc_auc": roc_auc_score(y_test, baseline_proba)
}

baseline_results


{'model': 'Baseline Random Forest',
 'accuracy': 0.9751332149200711,
 'precision': 0.9655172413793104,
 'recall': 0.8842105263157894,
 'f1_score': 0.9230769230769231,
 'roc_auc': 0.9982512370670265}

In [9]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "class_weight": ["balanced", "balanced_subsample"]
}

In [10]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 5, 10, 15, 20, 30],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": ["balanced", "balanced_subsample"]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__class_weight': ['balanced', 'balanced_subsample'], 'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 2, ...], ...}"
,n_iter,30
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [11]:
random_search.best_params_

{'model__n_estimators': 500,
 'model__min_samples_split': 5,
 'model__min_samples_leaf': 1,
 'model__max_features': 'sqrt',
 'model__max_depth': None,
 'model__class_weight': 'balanced_subsample'}

In [12]:
random_search.best_score_

np.float64(0.8372135991128001)

In [13]:
tuned_rf = random_search.best_estimator_

tuned_pred = tuned_rf.predict(X_test)
tuned_proba = tuned_rf.predict_proba(X_test)[:, 1]

tuned_results = {
    "model": "Tuned Random Forest",
    "accuracy": accuracy_score(y_test, tuned_pred),
    "precision": precision_score(y_test, tuned_pred),
    "recall": recall_score(y_test, tuned_pred),
    "f1_score": f1_score(y_test, tuned_pred),
    "roc_auc": roc_auc_score(y_test, tuned_proba)
}

tuned_results

{'model': 'Tuned Random Forest',
 'accuracy': 0.9680284191829485,
 'precision': 0.9184782608695652,
 'recall': 0.8894736842105263,
 'f1_score': 0.9037433155080213,
 'roc_auc': 0.9950910931174088}

In [14]:
tuning_comparison = pd.DataFrame([baseline_results, tuned_results])

tuning_comparison

,model,accuracy,precision,recall,f1_score,roc_auc
0,Baseline Random Forest,0.975133,0.965517,0.884211,0.923077,0.998251
1,Tuned Random Forest,0.968028,0.918478,0.889474,0.903743,0.995091


In [15]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, tuned_pred))

print("\nClassification Report:")
print(classification_report(y_test, tuned_pred))

Confusion Matrix:
[[921  15]
 [ 21 169]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       936
           1       0.92      0.89      0.90       190

    accuracy                           0.97      1126
   macro avg       0.95      0.94      0.94      1126
weighted avg       0.97      0.97      0.97      1126



In [18]:
os.makedirs("../../05_outputs/model_results", exist_ok=True)

tuning_comparison.to_csv(
    "../../05_outputs/model_results/random_forest_tuning_comparison.csv",
    index=False
)

cv_results_df = pd.DataFrame(random_search.cv_results_)

cv_results_df.to_csv(
    "../../05_outputs/model_results/random_forest_randomized_search_cv_results.csv",
    index=False
)

os.path.exists("../../05_outputs/model_results/random_forest_tuning_comparison.csv")

True

In [19]:
os.makedirs("../../06_models", exist_ok=True)

joblib.dump(
    tuned_rf,
    "../../06_models/tuned_random_forest_churn_model.pkl"
)

os.path.exists("../../06_models/tuned_random_forest_churn_model.pkl")

True

## Hyperparameter Tuning Summary

RandomizedSearchCV was used to tune the Random Forest model with 5-fold cross-validation. F1-score was selected as the optimization metric because the dataset is imbalanced and accuracy alone may not reflect the model's ability to identify churned customers.

The tuned Random Forest slightly improved recall compared with the baseline model. However, it reduced precision, F1-score, accuracy, and ROC-AUC on the test set.

Therefore, the baseline Random Forest remains the selected final model.

This result shows that hyperparameter tuning was tested, but the simpler baseline Random Forest provided a better overall balance between precision and recall on the test data.

The tuning outputs were saved to:

- `05_outputs/model_results/random_forest_tuning_comparison.csv`
- `05_outputs/model_results/random_forest_randomized_search_cv_results.csv`

The tuned model was also saved for reference:

- `06_models/tuned_random_forest_churn_model.pkl`